In [ ]:
val_preds = np.load('/global/homes/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default/val_preds_epoch_1.npy')
val_targets = np.load('/global/homes/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default/val_targets_epoch_1.npy')
val_res = val_targets - val_preds

val_res = val_res.reshape(-1, val_preds.shape[2])
val_preds = val_preds.reshape(-1, val_preds.shape[2])


In [ ]:
preds_std = val_preds.std(axis=0)
torch.save(torch.from_numpy(preds_std), '/global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default' + '/preds_std.pt')

preds_mean = val_preds.mean(axis=0)
torch.save(torch.from_numpy(preds_mean), '/global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default' + '/preds_mean.pt')

res_std = val_res.std(axis=0)
torch.save(torch.from_numpy(res_std), '/global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default' + '/res_std.pt')

res_mean = val_res.mean(axis=0)
torch.save(torch.from_numpy(res_mean), '/global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default' + '/res_mean.pt')

In [ ]:
train_preds_load = np.load('/global/homes/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default/train_preds_epoch_1.npy')
train_targets = np.load('/global/homes/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default/train_targets_epoch_1.npy')
train_res = train_targets - train_preds

train_res = train_res.reshape(-1, train_preds.shape[2])
train_preds = train_preds.reshape(-1, train_preds.shape[2])


In [ ]:
preds_std = train_preds.std(axis=0)
torch.save(torch.from_numpy(preds_std), '/global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default' + '/preds_std.pt')

preds_mean = train_preds.mean(axis=0)
torch.save(torch.from_numpy(preds_mean), '/global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default' + '/preds_mean.pt')

res_std = train_res.std(axis=0)
torch.save(torch.from_numpy(res_std), '/global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default' + '/res_std.pt')

res_mean = train_res.mean(axis=0)
torch.save(torch.from_numpy(res_mean), '/global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default' + '/res_mean.pt')

In [ ]:
#'/preds_mean.pt'

def reshape_tensor(path_end):
    preds_mean = torch.load('/global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default/' + path_end + '.pt')
    mean_profile = preds_mean[:data.target_profile_num*60]
    mean_scalar = preds_mean[data.target_profile_num*60:]
    
    # reshape x_profile to (batch, input_profile_num, levels)
    mean_profile = mean_profile.reshape(data.target_profile_num, 60)
    # broadcast x_scalar to (batch, input_scalar_num, levels)
    mean_scalar = mean_scalar.unsqueeze(1).expand( -1, 60)
    
    #concatenate x_profile, x_scalar, x_loc to (batch, input_profile_num+input_scalar_num, levels)
    preds_mean = torch.cat((mean_profile, mean_scalar), dim=0)
    
    # pads the beginning of levels so that levels = seq_resolution (which by default is 64)
    preds_mean = torch.nn.functional.pad(preds_mean, (4,0), "constant", 0.0)
    new_path = '/global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default/reshaped_'+ path_end + '.pt'
    torch.save(preds_mean, new_path)
    print(f'saved to {new_path}')